# UM Building Footprints GeoJSON Cleaner

### Part 1
- Loads the UM building footprints GeoJSON
- Simplifies geometries for web mapping performance
- Drops unneeded GIS-only metadata fields
- Creates and normalizes AI resource column into a boolean
- Outputs a cleaned geoJSON

In [2]:
import pandas as pd
import geopandas as gpd

In [6]:
# Load and inspect the data
building_footprint = gpd.read_file('um-building-footprint.geojson')
print("rows:", len(building_footprint))
print("crs:", building_footprint.crs) #Checks for coordinate reference system
print("geometry types:\n", building_footprint.geom_type.value_counts()) #Some multipolygons, will need to be converted to polygons
building_footprint.head()

rows: 1042
crs: EPSG:4326
geometry types:
 Polygon         1007
MultiPolygon      35
Name: count, dtype: int64


,OBJECTID,loc_Campus_descr,loc_SubCampus_descr,loc_MainCampus,Status,Ownership,PanelType,ObjectName,Building_Type,loc_ObjectNum,created_user,created_date,last_edited_user,last_edited_date,GlobalID,SUBTYPE,Shape__Area,Shape__Length,geometry
0,1,Ann Arbor,North Campus,Main,Active,Own,Building,NCRC B500 (Underground),BLDGU,1005280,brimcel_umich,1751976683021,brimcel_umich,1751976683021,5022191f-82a9-4b0d-9ae3-8fe1fac098cf,3,15083.046875,1214.576194,"POLYGON ((-83.70362 42.2999, -83.70362 42.2999..."
1,2,Ann Arbor,North Campus,Main,Active,Own,Building,NCRC Bldg 15,BLDGU,1005255,brimcel_umich,1751976683021,brimcel_umich,1751976683021,ff531986-1ca8-40a8-aff9-a3ae7b7670d9,3,5937.982910,322.868746,"POLYGON ((-83.7067 42.30023, -83.7067 42.30004..."
2,3,Ann Arbor,North Campus,Main,Active,Own,Building,NCRC Bldg 23,BLDGU,1005261,brimcel_umich,1751976683021,brimcel_umich,1751976683021,492056f7-f411-46da-8094-f323a85481a3,3,10717.561279,643.277263,"POLYGON ((-83.70674 42.29968, -83.70674 42.299..."
3,4,Ann Arbor,Central Campus,Main,Active,Own,Building,Central Campus Support Facility,BLDGU,1005379,brimcel_umich,1751976683021,brimcel_umich,1751976683021,3a10f37e-15f9-41cf-bfb6-8027c21d1bad,3,87.915283,39.349791,"POLYGON ((-83.7354 42.27905, -83.7354 42.27907..."
4,6,Ann Arbor,Medical Campus,Main,Active,Own,Building,Cardiovascular Center Parking Structure,PARKU,1005146,brimcel_umich,1751976683021,brimcel_umich,1751976683021,a314aa07-9fc4-40aa-9e73-18c9827a08e2,3,75928.744141,1362.263516,"POLYGON ((-83.73085 42.28281, -83.73082 42.282..."


In [7]:
# Dropping unnecessary columns
keep_cols = [
    "ObjectName",
    "loc_ObjectNum",
    "loc_Campus_descr",
    "loc_SubCampus_descr"
]

gdf = building_footprint[keep_cols + ["geometry"]].copy()
gdf.head(2)

,ObjectName,loc_ObjectNum,loc_Campus_descr,loc_SubCampus_descr,geometry
0,NCRC B500 (Underground),1005280,Ann Arbor,North Campus,"POLYGON ((-83.70362 42.2999, -83.70362 42.2999..."
1,NCRC Bldg 15,1005255,Ann Arbor,North Campus,"POLYGON ((-83.7067 42.30023, -83.7067 42.30004..."


In [8]:
# Rename columns to be cleaner and consistent/contextual
names = {
    "ObjectName": "building_name",
    "loc_ObjectNum": "building_id",
    "loc_Campus_descr": "campus",
    "loc_SubCampus_descr": "sub_campus",
}

gdf = gdf.rename(columns={k:v for k,v in names.items() if k in gdf.columns})
gdf.head(2)

,building_name,building_id,campus,sub_campus,geometry
0,NCRC B500 (Underground),1005280,Ann Arbor,North Campus,"POLYGON ((-83.70362 42.2999, -83.70362 42.2999..."
1,NCRC Bldg 15,1005255,Ann Arbor,North Campus,"POLYGON ((-83.7067 42.30023, -83.7067 42.30004..."


In [9]:
# Mark all AI resources as True in the AI_resource column, and False for all other buildings
resource_list = [
    "Vaughan SPH-I", #SABER
    "Institute for Social Research", #d3 ---- #ESC
    "NCRC Bldg 550", #DISC
    "NCRC Bldg 520", #e-HAIL
    "NCRC Bldg 100", #AI & Digital Health Innovation ---- #Center for Global Health Equity
    "Beyster Building", #AI Lab
    "3520 Green Court", #Institute for Comp. Discovery and Engineering
    "Alexander G. Ruthven Building", #ARIA ---- #Bold Challenges
    "East Hall", #Center for Applied and Interdisc. Mathematics
    "Mason Hall", #Digital Studies Institute
    "Weill Hall", #STPP Program
    "Weiser Hall" #MIDAS
]

# Still to find a footprint for:
# ARC
# Alzeimer's Disease Research Center
# AIIM

gdf["is_resource"] = gdf["building_name"].isin(resource_list)
gdf.head()

,building_name,building_id,campus,sub_campus,geometry,is_resource
0,NCRC B500 (Underground),1005280,Ann Arbor,North Campus,"POLYGON ((-83.70362 42.2999, -83.70362 42.2999...",False
1,NCRC Bldg 15,1005255,Ann Arbor,North Campus,"POLYGON ((-83.7067 42.30023, -83.7067 42.30004...",False
2,NCRC Bldg 23,1005261,Ann Arbor,North Campus,"POLYGON ((-83.70674 42.29968, -83.70674 42.299...",False
3,Central Campus Support Facility,1005379,Ann Arbor,Central Campus,"POLYGON ((-83.7354 42.27905, -83.7354 42.27907...",False
4,Cardiovascular Center Parking Structure,1005146,Ann Arbor,Medical Campus,"POLYGON ((-83.73085 42.28281, -83.73082 42.282...",False


In [10]:
# Check that all the correct buildings are marked as true
gdf[gdf["is_resource"] == True]

,building_name,building_id,campus,sub_campus,geometry,is_resource
42,Beyster Building,1005092,Ann Arbor,North Campus,"POLYGON ((-83.71612 42.29335, -83.71612 42.293...",True
268,Mason Hall,1000197,Ann Arbor,Central Campus,"POLYGON ((-83.7393 42.2774, -83.7393 42.2774, ...",True
271,East Hall,1000166,Ann Arbor,Central Campus,"MULTIPOLYGON (((-83.73479 42.27616, -83.73479 ...",True
454,Vaughan SPH-I,1000204,Ann Arbor,Central Campus,"POLYGON ((-83.73001 42.28122, -83.73001 42.281...",True
495,NCRC Bldg 550,1005282,Ann Arbor,North Campus,"POLYGON ((-83.70149 42.30019, -83.7013 42.3001...",True
496,NCRC Bldg 520,1005281,Ann Arbor,North Campus,"POLYGON ((-83.70253 42.30066, -83.70253 42.300...",True
500,NCRC Bldg 100,1005276,Ann Arbor,North Campus,"POLYGON ((-83.70305 42.30118, -83.70305 42.301...",True
535,Institute for Social Research,1000145,Ann Arbor,Central Campus,"POLYGON ((-83.74359 42.27716, -83.74359 42.277...",True
563,Weill Hall,1005101,Ann Arbor,Central Campus,"POLYGON ((-83.74002 42.27271, -83.74002 42.272...",True
598,Weiser Hall,1000165,Ann Arbor,Central Campus,"MULTIPOLYGON (((-83.73511 42.27691, -83.73511 ...",True


In [11]:
# Output finalized geoJSON
gdf.to_file("um-building-footprint-edited.geojson", driver="GeoJSON")

### Part 2

- Loads the CSV file containing resource details for map pop-ups and preps for merging
- Merges the building footprint and resource data files using building_id as the key

In [16]:
# Load in resource data
resources = pd.read_csv("resources.csv")

# Convert building_id to string for merging
resources["building_id"] = resources["building_id"].astype(str)

In [24]:
# Merge the resource data with the building footprint data and check if the merge worked correctly
final_gdf = gdf.merge(resources, left_on="building_id", right_on="building_id", how="left")
final_gdf[final_gdf["is_resource"] == True][["building_name", "building_id", "name"]]

,building_name,building_id,name
42,Beyster Building,1005092,NaN
268,Mason Hall,1000197,NaN
271,East Hall,1000166,NaN
454,Vaughan SPH-I,1000204,NaN
495,NCRC Bldg 550,1005282,NaN
496,NCRC Bldg 520,1005281,NaN
500,NCRC Bldg 100,1005276,NaN
535,Institute for Social Research,1000145,NaN
563,Weill Hall,1005101,NaN
598,Weiser Hall,1000165,NaN
